# Lab 2 - Comitês (Ensembles)

Classificadores: **Random Forest**, **AdaBoost** e **XGBoost**.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime

from lab2_utils import avaliar_modelo, logar_mlflow, iniciar_run

## Carregamento dos Dados Processados

In [2]:
data_dir = './dados_processados'

X_train = pd.read_parquet(f'{data_dir}/X_train.parquet')
y_train = pd.read_parquet(f'{data_dir}/y_train.parquet').values.ravel()
X_test = pd.read_parquet(f'{data_dir}/X_test.parquet')
y_test = pd.read_parquet(f'{data_dir}/y_test.parquet').values.ravel()

mod_time = datetime.fromtimestamp(os.path.getmtime(f'{data_dir}/X_train.parquet'))
print(f'Dados carregados (gerados em {mod_time.strftime("%Y-%m-%d %H:%M")})')
print(f'  X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'  y_train: {y_train.shape}  |  y_test: {y_test.shape}')
print(f'  Features: {list(X_train.columns)}')
print(f'  Churn rate treino: {y_train.mean():.3f}  |  teste: {y_test.mean():.3f}')

Dados carregados (gerados em 2026-06-24 15:43)
  X_train: (440832, 11)  |  X_test: (64374, 11)
  y_train: (440832,)  |  y_test: (64374,)
  Features: ['Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Contract Length', 'Total Spend', 'Last Interaction', 'Subscription Type_Premium', 'Subscription Type_Standard']
  Churn rate treino: 0.567  |  teste: 0.474


## Configuração do MLflow

O módulo `lab2_utils.py` centraliza toda a configuração. Ao importá-lo, ele:

1. **Inicia o servidor MLflow automaticamente** (se não estiver rodando)
2. Configura tracking via HTTP (`http://127.0.0.1:5000`)
3. Garante runs idempotentes (re-executar substitui o run anterior)

A UI fica disponível em **http://127.0.0.1:5000** — basta abrir no navegador.

In [3]:
# Funções avaliar_modelo() e logar_mlflow() vêm do lab2_utils.py
# Nada a definir aqui — célula mantida para compatibilidade de estrutura.
print("MLflow configurado via lab2_utils.py")
print(f"  Experimento: Lab2-Churn")
print(f"  Tracking URI: sqlite:///mlflow.db")

MLflow configurado via lab2_utils.py
  Experimento: Lab2-Churn
  Tracking URI: sqlite:///mlflow.db


## Nota sobre Normalização

Os dados já vêm escalonados (`StandardScaler`) do notebook 1. Modelos baseados em árvores são invariantes a escalonamento, então isso não afeta os resultados.

## Exemplo de Uso — Padrão MLflow

Comitês não precisam de scaler, então usamos `X_train`/`X_test` diretamente.
O `run_name` é fixo (nome do modelo) — cada execução gera um novo run diferenciado pelo horário e features na UI.

```python
params = {'modelo': 'RandomForest', 'n_estimators': 100, ...}
with iniciar_run("RandomForest", notebook="2C", params=params):
    model.fit(...)
    metricas = avaliar_modelo('RandomForest', y_test, y_pred)
    logar_mlflow(metricas, params, artefatos=[fi_path])
```

In [ ]:
# ─── Random Forest (v4) ─────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier

n_est = 300
params = {
    'modelo': 'RandomForest',
    'n_estimators': n_est,
    'max_depth': 30,
    'min_samples_split': 5,
    'min_samples_leaf': 2,
    'scaler': 'nenhum (dados já escalados)',
}

with iniciar_run("RandomForest", notebook="2C", params=params):
    model = RandomForestClassifier(
        n_estimators=n_est,
        max_depth=30,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    metricas = avaliar_modelo('RandomForest', y_test, y_pred)

    fig, ax = plt.subplots(figsize=(10, 6))
    importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values()
    importances.plot.barh(ax=ax, color='steelblue', edgecolor='white')
    ax.set_title('Feature Importance — Random Forest')
    plt.tight_layout()
    fi_path = '../relatorio/imagens/2c_com_feature_importance_rf.png'
    plt.savefig(fi_path, dpi=150, bbox_inches='tight')
    plt.show()

    logar_mlflow(metricas, params, artefatos=[fi_path])